# 1. Load the Libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 2. Load the Dataset 

In [3]:
df = pd.read_csv('D:\\Sales_Dashboard\\Sales_Raw_Data.csv')
df.head()

,Order_ID,Order_Date,Customer_ID,Customer_Name,Age,Gender,City,Product,Category,Quantity,Unit_Price,Total_Sales
0,ORD100002,2025-02-25,CUST5529,Customer_227,30.0,Female,Bengaluru,Rice,Grocery,7,2829.77,19808.39
1,ORD100003,2025-10-14,CUST3127,Customer_182,63.0,Male,Bengaluru,Book,Education,5,27906.16,139530.80
2,ORD100004,2025-05-13,CUST8887,Customer_487,62.0,Female,Bengaluru,Book,Education,8,37491.06,299928.48
3,ORD100005,2025-12-02,CUST2515,Customer_470,65.0,Female,Kolkata,Mobile,Electronics,9,28541.36,256872.24
4,ORD100006,2025-11-20,CUST4796,Customer_380,44.0,Male,Bengaluru,Rice,Grocery,10,14036.59,140365.90


### 2.1 Information of Dataset

In [13]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Order_ID       1000 non-null   object 
 1   Order_Date     1000 non-null   object 
 2   Customer_ID    1000 non-null   object 
 3   Customer_Name  1000 non-null   object 
 4   Age            980 non-null    float64
 5   Gender         1000 non-null   object 
 6   City           987 non-null    object 
 7   Product        1000 non-null   object 
 8   Category       1000 non-null   object 
 9   Quantity       1000 non-null   int64  
 10  Unit_Price     1000 non-null   float64
 11  Total_Sales    1000 non-null   float64
dtypes: float64(3), int64(1), object(8)
memory usage: 93.9+ KB


### 2.2 Creating Data Dictionary 

In [14]:
data_dictionary = pd.DataFrame([
    ["Order_ID",      "Text (ID)",  "Unique identifier for each order",                              "Used to detect duplicate/repeated orders"],
    ["Order_Date",     "Date",       "Date when the order was placed",                                "Enables trend & seasonality analysis"],
    ["Customer_ID",    "Text (ID)",  "Unique identifier for each customer",                            "Used to track repeat customers"],
    ["Customer_Name",  "Text",       "Name of the customer",                                          "Useful for reporting"],
    ["Age",            "Numeric",    "Age of the customer in years",                                  "Enables age-group based segmentation"],
    ["Gender",         "Category",   "Gender of the customer (Male/Female)",                          "Useful for demographic analysis"],
    ["City",           "Category",   "City where the customer is located",                            "Enables regional sales analysis"],
    ["Product",        "Category",   "Name of the product purchased",                                 "Identifies best-selling products"],
    ["Category",       "Category",   "Product category (e.g., Electronics, Fashion)",                 "Enables category-level sales analysis"],
    ["Quantity",       "Numeric",    "Number of units purchased in the order",                        "Used to calculate sales volume"],
    ["Unit_Price",     "Numeric",    "Price per unit of the product (INR)",                            "Used to calculate revenue"],
    ["Total_Sales",    "Numeric",    "Total value of the order (Quantity x Unit_Price)",              "Core revenue metric for the business"],
], columns=["Column", "Data Type", "Description", "Business Relevance"])

data_dictionary


,Column,Data Type,Description,Business Relevance
0,Order_ID,Text (ID),Unique identifier for each order,Used to detect duplicate/repeated orders
1,Order_Date,Date,Date when the order was placed,Enables trend & seasonality analysis
2,Customer_ID,Text (ID),Unique identifier for each customer,Used to track repeat customers
3,Customer_Name,Text,Name of the customer,Useful for reporting
4,Age,Numeric,Age of the customer in years,Enables age-group based segmentation
5,Gender,Category,Gender of the customer (Male/Female),Useful for demographic analysis
6,City,Category,City where the customer is located,Enables regional sales analysis
7,Product,Category,Name of the product purchased,Identifies best-selling products
8,Category,Category,"Product category (e.g., Electronics, Fashion)",Enables category-level sales analysis
9,Quantity,Numeric,Number of units purchased in the order,Used to calculate sales volume


### 2.3 Save the File in the System

In [16]:
data_dictionary.to_csv(r"D:\\Sales_Dashboard\\Data_Dictionary.csv", index=False)
print("File Saved Successfully")

File Saved Successfully


# 3. Data Quality Check

### 3.1 Missing Values 

In [7]:
missing_summary = df.isnull().sum()
missing_pct = (missing_summary / len(df) * 100).round(2)

missing_data = pd.DataFrame({ "Missing Count": missing_summary, "Missing %": missing_pct})
missing_report = missing_data[missing_data["Missing Count"] > 0].sort_values("Missing Count", ascending=False)

print("Columns with Missing Values:")
missing_report


Columns with Missing Values:


,Missing Count,Missing %
Age,20,2.0
City,13,1.3


## 3.2 Duplicate Records

In [12]:
All_duplicates = df.duplicated().sum()
order_id_duplicate = df.duplicated(subset=["Order_ID"]).sum()

print(f"Fully duplicated rows: {All_duplicates}")
print(f"Duplicate Order id :   {order_id_duplicate}")

dup_order_ids = df[df.duplicated(subset=["Order_ID"], keep=False)].sort_values("Order_ID")
dup_order_ids[["Order_ID", "Customer_ID", "Order_Date","Quantity", "Total_Sales"]]


Fully duplicated rows: 0
Duplicate Order id :   8


,Order_ID,Customer_ID,Order_Date,Quantity,Total_Sales
48,ORD100050,CUST5523,2025-10-03,9,277513.83
118,ORD100050,CUST3194,2025-01-22,2,51591.10
238,ORD100050,CUST8467,2025-03-01,3,8582.49
358,ORD100050,CUST6467,2025-02-11,7,318362.73
478,ORD100050,CUST8225,2025-11-22,8,311660.72
598,ORD100050,CUST8859,2025-02-21,7,141013.88
718,ORD100050,CUST7824,2025-11-06,4,170570.36
838,ORD100050,CUST5585,2025-08-03,9,258727.23
958,ORD100050,CUST7188,2025-02-09,1,34619.77


### 3.3 Inconsistent Formating

In [18]:
categorical_cols = ["Gender", "City", "Product", "Category"]

for col in categorical_cols:
    print(f"--- {col} ---")
    print(sorted(df[col].dropna().unique()))
    print()


--- Gender ---
['Female', 'Male']

--- City ---
['Bengaluru', 'Delhi', 'Gaya', 'Hyderabad', 'Kolkata', 'Mumbai', 'Patna', 'Pune']

--- Product ---
['Book', 'Chair', 'Laptop', 'Mobile', 'Rice', 'Shoes']

--- Category ---
['Education', 'Electronics', 'Fashion', 'Furniture', 'Grocery']



In [20]:
# Check the date column format consistency
print("Sample of Order_Date values: ")
print(df["Order_Date"].head())



Sample of Order_Date values: 
0    2025-02-25
1    2025-10-14
2    2025-05-13
3    2025-12-02
4    2025-11-20
Name: Order_Date, dtype: object


# 4. Data Cleaning & Transformation

### 4.1 Remove Duplicate Order_Id

In [21]:
before = len(df)
df = df.drop_duplicates(subset=["Order_ID"], keep="first")
after = len(df)

print(f"Removed {before - after} duplicate Order_ID rows")
print(f"New shape: {df.shape}")

assert df["Order_ID"].is_unique, "Order_ID still has Duplicate Records!"
print("Order_ID is now unique")


Removed 8 duplicate Order_ID rows
New shape: (992, 12)
Order_ID is now unique


### 4.2 Handle Missing Values

#### Age column consist of missing values fill with median age

In [24]:
median_age = df["Age"].median()
df["Age"] = df["Age"].fillna(median_age)
print(f"Filled missing Age values with median = {median_age}")

Filled missing Age values with median = 41.0


#### City Column consist of missing values fill with unknown 

In [25]:
df["City"] = df["City"].fillna("Unknown")
print("Filled missing City values with 'Unknown'")

Filled missing City values with 'Unknown'


In [26]:
print("Remaining missing values:")
print(df.isnull().sum())

Remaining missing values:
Order_ID         0
Order_Date       0
Customer_ID      0
Customer_Name    0
Age              0
Gender           0
City             0
Product          0
Category         0
Quantity         0
Unit_Price       0
Total_Sales      0
dtype: int64


### 4.3 Standardize Date Format

In [28]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"], format="%Y-%m-%d")

print("Order_Date dtype is now:", df["Order_Date"].dtype)
df[["Order_Date"]].head()


Order_Date dtype is now: datetime64[ns]


,Order_Date
0,2025-02-25
1,2025-10-14
2,2025-05-13
3,2025-12-02
4,2025-11-20


### 4.4 Standardize Text Format

In [29]:
text_cols = ["Customer_Name", "Gender", "City", "Product", "Category"]

for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.title()

df[text_cols].head()


,Customer_Name,Gender,City,Product,Category
0,Customer_227,Female,Bengaluru,Rice,Grocery
1,Customer_182,Male,Bengaluru,Book,Education
2,Customer_487,Female,Bengaluru,Book,Education
3,Customer_470,Female,Kolkata,Mobile,Electronics
4,Customer_380,Male,Bengaluru,Rice,Grocery


### 4.5 Categorize Age column To Readable Age_Groups

In [32]:
age_bins   = [0, 25, 35, 45, 55, 100]
age_labels = ["18-25", "26-35", "36-45", "46-55", "56+"]

df["Age_Group"] = pd.cut(df["Age"], bins=age_bins, labels=age_labels)

df[["Age", "Age_Group"]].head()


,Age,Age_Group
0,30.0,26-35
1,63.0,56+
2,62.0,56+
3,65.0,56+
4,44.0,36-45


### 4.6 Feature Engineering from Order_Date

In [33]:
df["Order_Year"]    = df["Order_Date"].dt.year
df["Order_Month"]   = df["Order_Date"].dt.month_name()
df["Order_Weekday"] = df["Order_Date"].dt.day_name()
df["Order_Quarter"] = df["Order_Date"].dt.quarter

df[["Order_Date", "Order_Year", "Order_Month", "Order_Weekday", "Order_Quarter"]].head()


,Order_Date,Order_Year,Order_Month,Order_Weekday,Order_Quarter
0,2025-02-25,2025,February,Tuesday,1
1,2025-10-14,2025,October,Tuesday,4
2,2025-05-13,2025,May,Tuesday,2
3,2025-12-02,2025,December,Tuesday,4
4,2025-11-20,2025,November,Thursday,4


### 4.7 Check Of Cleaned Data

In [34]:
print("Final shape:", df.shape)
print()
print("Missing values per column:")
print(df.isnull().sum())
print()
df.info()


Final shape: (992, 17)

Missing values per column:
Order_ID         0
Order_Date       0
Customer_ID      0
Customer_Name    0
Age              0
Gender           0
City             0
Product          0
Category         0
Quantity         0
Unit_Price       0
Total_Sales      0
Age_Group        0
Order_Year       0
Order_Month      0
Order_Weekday    0
Order_Quarter    0
dtype: int64

<class 'pandas.core.frame.DataFrame'>
Index: 992 entries, 0 to 999
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Order_ID       992 non-null    object        
 1   Order_Date     992 non-null    datetime64[ns]
 2   Customer_ID    992 non-null    object        
 3   Customer_Name  992 non-null    object        
 4   Age            992 non-null    float64       
 5   Gender         992 non-null    object        
 6   City           992 non-null    object        
 7   Product        992 non-null    object        
 8

In [35]:
# Displaying First 5 rows of data
df.head()

,Order_ID,Order_Date,Customer_ID,Customer_Name,Age,Gender,City,Product,Category,Quantity,Unit_Price,Total_Sales,Age_Group,Order_Year,Order_Month,Order_Weekday,Order_Quarter
0,ORD100002,2025-02-25,CUST5529,Customer_227,30.0,Female,Bengaluru,Rice,Grocery,7,2829.77,19808.39,26-35,2025,February,Tuesday,1
1,ORD100003,2025-10-14,CUST3127,Customer_182,63.0,Male,Bengaluru,Book,Education,5,27906.16,139530.80,56+,2025,October,Tuesday,4
2,ORD100004,2025-05-13,CUST8887,Customer_487,62.0,Female,Bengaluru,Book,Education,8,37491.06,299928.48,56+,2025,May,Tuesday,2
3,ORD100005,2025-12-02,CUST2515,Customer_470,65.0,Female,Kolkata,Mobile,Electronics,9,28541.36,256872.24,56+,2025,December,Tuesday,4
4,ORD100006,2025-11-20,CUST4796,Customer_380,44.0,Male,Bengaluru,Rice,Grocery,10,14036.59,140365.90,36-45,2025,November,Thursday,4


### 5. Export The Cleaned Data File

In [36]:
df.to_csv(r"D:\\Sales_Dashboard\\Sales_Cleaned_Data.csv", index=False)
print("File Saved Successfully")
print('Final shape:', df.shape)
df.head()

File Saved Successfully
Final shape: (992, 17)


,Order_ID,Order_Date,Customer_ID,Customer_Name,Age,Gender,City,Product,Category,Quantity,Unit_Price,Total_Sales,Age_Group,Order_Year,Order_Month,Order_Weekday,Order_Quarter
0,ORD100002,2025-02-25,CUST5529,Customer_227,30.0,Female,Bengaluru,Rice,Grocery,7,2829.77,19808.39,26-35,2025,February,Tuesday,1
1,ORD100003,2025-10-14,CUST3127,Customer_182,63.0,Male,Bengaluru,Book,Education,5,27906.16,139530.80,56+,2025,October,Tuesday,4
2,ORD100004,2025-05-13,CUST8887,Customer_487,62.0,Female,Bengaluru,Book,Education,8,37491.06,299928.48,56+,2025,May,Tuesday,2
3,ORD100005,2025-12-02,CUST2515,Customer_470,65.0,Female,Kolkata,Mobile,Electronics,9,28541.36,256872.24,56+,2025,December,Tuesday,4
4,ORD100006,2025-11-20,CUST4796,Customer_380,44.0,Male,Bengaluru,Rice,Grocery,10,14036.59,140365.90,36-45,2025,November,Thursday,4
